#### Data Cleaning

In [1]:
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("../data/processed/ml_dataset.csv")

In [3]:
df.head()

,product_id,product_name,year,month,month_number,monthly_revenue,unique_customers,units_per_customer,revenue_per_customer,next_monthly_revenue,...,pct_transactions_discount_gt_10,pct_transactions_discount_gt_15,average_selling_price,minimum_selling_price,maximum_selling_price,price_range,non_discounted_revenue,discounted_revenue,discounted_quantity_sold,discounted_orders
0,P0001,Premium Rice 500g,2021,January,1,1960.47,6.0,6.0,326.75,1248.72,...,0.0,0.0,326.75,96.10,485.76,389.66,0.0,1960.47,40,6
1,P0001,Premium Rice 500g,2021,February,2,1248.72,4.0,6.0,312.18,3088.80,...,0.0,0.0,312.18,47.52,528.00,480.48,528.0,720.72,14,3
2,P0001,Premium Rice 500g,2021,March,3,3088.80,9.0,6.0,343.20,593.46,...,0.0,0.0,343.20,48.05,528.00,479.95,528.0,2560.80,52,8
3,P0001,Premium Rice 500g,2021,April,4,593.46,5.0,2.0,118.69,2042.83,...,0.0,0.0,118.69,49.10,152.06,102.96,0.0,593.46,12,5
4,P0001,Premium Rice 500g,2021,May,5,2042.83,7.0,5.0,291.83,1021.69,...,0.0,0.0,291.83,105.60,512.16,406.56,528.0,1514.83,30,5


In [4]:
df.isna().sum()

product_id                             0
product_name                           0
year                                   0
month                                  0
month_number                           0
monthly_revenue                        0
unique_customers                     191
units_per_customer                   191
revenue_per_customer                 191
next_monthly_revenue                1820
is_quarter_end                         0
is_year_end                            0
trend_year                             0
sin_month                              0
cos_month                              0
monthly_quantity                       0
monthly_transactions                   0
lag_1_revenue                       1820
lag_2_revenue                       3640
lag_3_revenue                       5460
lag_12_revenue                     21840
lag_1_quantity                      1820
lag_2_quantity                      3640
lag_3_quantity                      5460
lag_12_quantity 

In [5]:
for col in df.columns:
    if df[col].isna().sum() == 191:
        df.fillna({col: 0}, inplace=True)

In [6]:
for col in df.columns:
    if df[col].isna().sum() != 0:
        print(col, df[col].isna().sum())

next_monthly_revenue 1820
lag_1_revenue 1820
lag_2_revenue 3640
lag_3_revenue 5460
lag_12_revenue 21840
lag_1_quantity 1820
lag_2_quantity 3640
lag_3_quantity 5460
lag_12_quantity 21840
lag_1_transactions 1820
lag_2_transactions 3640
lag_3_transactions 5460
lag_12_transactions 21840
moving_avg_3_month 3640
moving_avg_6_month 9100
moving_avg_12_month 20020
month_over_month 2010
year_on_year 21994
moving_stdev_3_month 3640
moving_stdev_6_month 9100
momentum 5462


In [7]:
filtered_df = df[(df['year'] == 2021) | (df['year'] == 2022) | (df['year'] == 2023)]

In [8]:
filtered_df.shape

(65520, 54)

In [9]:
df_cleaned = filtered_df.dropna()
df_cleaned.shape

(43534, 54)

In [10]:
df_cleaned.isna().sum().sum()

np.int64(0)

In [11]:
df_cleaned['year'].value_counts().sort_index()

year
2022    21764
2023    21770
Name: count, dtype: int64

#### Recursive Feature Elimination with Cross Validation (RFECV)

In [12]:
from sklearn.datasets import make_regression
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline

In [13]:
X = df_cleaned.drop(['product_id', 'product_name', 'month', 'next_monthly_revenue'], axis=1)
y = df_cleaned['next_monthly_revenue']

In [14]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [15]:
model = RandomForestRegressor(n_estimators=100,random_state=42, n_jobs=-1)
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

In [16]:
rfecv = RFECV(estimator=model, step=2, cv=cv_strategy, scoring='neg_mean_squared_error', n_jobs=-1)
rfecv.fit(X_scaled, y)

,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,RandomForestR...ndom_state=42)
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",2
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionadded:: 0.18",-1
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
"cv_results_ cv_results_: dict of ndarraysAll arrays (values of the dictionary) are sorted in ascending orderby the number of features used (i.e., the first element of the arrayrepresents the models that used the least number of features, while thelast element represents the models that used all available features)... versionadded:: 1.0This dictionary contains the following keys:split(k)_test_score : ndarray of shape (n_subsets_of_features,) The cross-validation scores across (k)th fold.mean_test_score : ndarray of shape (n_subsets_of_features,) Mean of scores over the folds.std_test_score : ndarray of shape (n_subsets_of_features,) Standard deviation of scores over the folds.n_features : ndarray of shape (n_subsets_of_features,) Number of features used at

In [17]:
print(f"Optimal number of features: {rfecv.n_features_}")

Optimal number of features: 38


In [18]:
rfecv_results = pd.DataFrame({
    'Features': X.columns,
    'Selected': rfecv.support_,
    'Ranking': rfecv.ranking_,
    'n_features': rfecv.n_features_
}).sort_values(by='Ranking')

In [19]:
rfecv.cv_results_['mean_test_score']

array([-4.31614045e+09, -3.46593729e+09, -3.13835456e+09, -3.17443517e+09,
       -3.12338157e+09, -3.07602271e+09, -3.05609819e+09, -3.06319502e+09,
       -3.03105699e+09, -3.06609563e+09, -3.06078043e+09, -3.07050389e+09,
       -3.04801317e+09, -3.01194049e+09, -3.01198417e+09, -3.00164781e+09,
       -2.99893333e+09, -3.00157522e+09, -2.98992954e+09, -2.98734684e+09,
       -3.00259289e+09, -2.99584271e+09, -2.99534030e+09, -2.99335080e+09,
       -2.99125603e+09, -3.00553885e+09])

In [20]:
pd.DataFrame({
    "n_features": rfecv.cv_results_["n_features"],
    "mean_test_score": rfecv.cv_results_["mean_test_score"],
    "std_test_score": rfecv.cv_results_["std_test_score"]
})

,n_features,mean_test_score,std_test_score
0,1,-4.316140e+09,2.546662e+08
1,2,-3.465937e+09,1.907216e+08
2,4,-3.138355e+09,1.441175e+08
3,6,-3.174435e+09,1.471188e+08
4,8,-3.123382e+09,1.315474e+08
5,10,-3.076023e+09,1.448580e+08
6,12,-3.056098e+09,1.232668e+08
7,14,-3.063195e+09,1.304516e+08
8,16,-3.031057e+09,1.576732e+08
9,18,-3.066096e+09,1.349655e+08


In [21]:
X.columns[rfecv.support_]

Index(['month_number', 'monthly_revenue', 'revenue_per_customer', 'sin_month',
       'cos_month', 'monthly_quantity', 'lag_1_revenue', 'lag_2_revenue',
       'lag_3_revenue', 'lag_12_revenue', 'lag_1_quantity', 'lag_2_quantity',
       'lag_3_quantity', 'lag_12_quantity', 'lag_1_transactions',
       'lag_2_transactions', 'lag_3_transactions', 'lag_12_transactions',
       'moving_avg_3_month', 'moving_avg_6_month', 'moving_avg_12_month',
       'month_over_month', 'year_on_year', 'moving_stdev_3_month',
       'moving_stdev_6_month', 'momentum', 'total_profit', 'average_profit',
       'average_discount_pct', 'maximum_discount', 'total_discount_amount',
       'pct_transactions_discount_gt_10', 'average_selling_price',
       'minimum_selling_price', 'maximum_selling_price', 'price_range',
       'non_discounted_revenue', 'discounted_revenue'],
      dtype='str')

In [22]:
rfecv_results

,Features,Selected,Ranking,n_features
1,month_number,True,1,38
2,monthly_revenue,True,1,38
5,revenue_per_customer,True,1,38
15,lag_3_revenue,True,1,38
13,lag_1_revenue,True,1,38
11,monthly_quantity,True,1,38
10,cos_month,True,1,38
9,sin_month,True,1,38
14,lag_2_revenue,True,1,38
22,lag_2_transactions,True,1,38


#### Locking the Final Features

In [23]:
rfecv.support_

array([False,  True,  True, False, False,  True, False, False, False,
        True,  True,  True, False,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False, False,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True, False, False])

In [24]:
X[X.columns[rfecv.support_]]

,month_number,monthly_revenue,revenue_per_customer,sin_month,cos_month,monthly_quantity,lag_1_revenue,lag_2_revenue,lag_3_revenue,lag_12_revenue,...,average_discount_pct,maximum_discount,total_discount_amount,pct_transactions_discount_gt_10,average_selling_price,minimum_selling_price,maximum_selling_price,price_range,non_discounted_revenue,discounted_revenue
12,1,1743.45,174.35,5.000000e-01,8.660254e-01,35,1667.43,313.63,1604.59,1960.47,...,5.00,9,104.55,0.00,174.35,49.10,485.76,436.66,52.8,1690.65
13,2,1701.74,283.62,8.660254e-01,5.000000e-01,34,1743.45,1667.43,313.63,1248.72,...,5.33,10,93.46,0.00,283.62,147.31,512.16,364.85,264.0,1437.74
14,3,1293.07,431.02,1.000000e+00,6.120000e-17,26,1701.74,1743.45,1667.43,3088.80,...,5.67,10,79.73,0.00,431.02,347.42,475.20,127.78,0.0,1293.07
15,4,1601.95,400.49,8.660254e-01,-5.000000e-01,31,1293.07,1701.74,1743.45,593.46,...,2.00,5,34.85,0.00,400.49,362.21,451.44,89.23,422.4,1179.55
16,5,3174.87,317.49,5.000000e-01,-8.660254e-01,64,1601.95,1293.07,1701.74,2042.83,...,5.10,10,204.33,0.00,317.49,104.54,501.60,397.06,158.4,3016.47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109171,8,69930.00,9990.00,-8.660254e-01,-5.000000e-01,11,58117.50,63045.00,43875.00,62437.50,...,6.14,11,4320.00,14.29,9990.00,6075.00,13365.00,7290.00,0.0,69930.00
109172,9,71482.50,8935.31,-1.000000e+00,-1.840000e-16,12,69930.00,58117.50,63045.00,88627.50,...,11.25,14,9517.50,62.50,8935.31,5805.00,11880.00,6075.00,0.0,71482.50
109173,10,44550.00,7425.00,-8.660254e-01,5.000000e-01,7,71482.50,69930.00,58117.50,43605.00,...,6.50,11,2700.00,16.67,7425.00,6007.50,13365.00,7357.50,0.0,44550.00
109174,11,74655.00,10665.00,-5.000000e-01,8.660254e-01,12,44550.00,71482.50,69930.00,90855.00,...,7.29,13,6345.00,28.57,10665.00,6210.00,13230.00,7020.00,6750.0,67905.00


In [25]:
df = df.dropna(axis=0)
df.shape

(85244, 54)

In [26]:
df_train = df[(df['year'] == 2021) | (df['year'] == 2022) | (df['year'] == 2023)]
df_valid = df[(df['year'] == 2024)]
df_test = df[df['year'] == 2025]

In [27]:
df_train.shape

(43534, 54)

In [28]:
y_train = df_train['next_monthly_revenue']
y_valid = df_valid['next_monthly_revenue']
y_test = df_test['next_monthly_revenue']
print(y_train.shape, y_valid.shape, y_test.shape)

(43534,) (21762,) (19948,)


In [29]:
df_train = df_train.drop(['product_id', 'product_name', 'month', 'next_monthly_revenue'], axis=1)
df_valid = df_valid.drop(['product_id', 'product_name', 'month', 'next_monthly_revenue'], axis=1)
df_test = df_test.drop(['product_id', 'product_name', 'month', 'next_monthly_revenue'], axis=1)
print(df_train.shape, df_valid.shape, df_test.shape)

(43534, 50) (21762, 50) (19948, 50)


In [30]:
final_features = X.columns[rfecv.support_]

In [31]:
X_train = df_train[final_features]
X_valid = df_valid[final_features]
X_test = df_test[final_features]
print(X_train.shape, X_valid.shape, X_test.shape)

(43534, 38) (21762, 38) (19948, 38)


In [32]:
train_csv = pd.concat([X_train, y_train], axis=1)
valid_csv = pd.concat([X_valid, y_valid], axis=1)
test_csv = pd.concat([X_test, y_test], axis=1)

In [33]:
train_csv.shape, test_csv.shape, valid_csv.shape

((43534, 39), (19948, 39), (21762, 39))

In [34]:
train_csv.to_csv("../data/ml_ready/train.csv", index=False)
test_csv.to_csv("../data/ml_ready/test.csv", index=False)
valid_csv.to_csv("../data/ml_ready/valid.csv", index=False)

#### Saving Final Candidate Features Dataset

In [35]:
import pandas as pd
import numpy as np

In [36]:
df_train = pd.read_csv("../data/ml_ready/train.csv")
df_valid = pd.read_csv("../data/ml_ready/valid.csv")
df_test = pd.read_csv("../data/ml_ready/test.csv")